# Logging models with MLFLow

Many issues arise when using MLflow with Azure-mlflow. According to Gemini this might be due to a Flask dependency and the API whihc has changed in MLFLow >= 3.0.0. 
Using MLFlow skinny according to this [source](https://github.com/azure/azure-sdk-for-python/issues/44365) might solve this. 

In [1]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
import dotenv
import os

dotenv.load_dotenv(override=True)

ml_client = MLClient(
     credential         = DefaultAzureCredential()
    ,subscription_id    = os.environ["AZURE_SUBSCRIPTION_ID"]
    ,resource_group_name= os.environ["AZURE_RESOURCE_GROUP"]
    ,workspace_name     = os.environ["AZURE_ML"]
)


Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


In [2]:
import mlflow

ml_tracking_uri = ml_client.workspaces.get(os.environ["AZURE_ML"]).mlflow_tracking_uri
mlflow.set_tracking_uri(ml_tracking_uri)

print(f"{mlflow.get_tracking_uri()[:15]}****************")

azureml://weste****************


In [3]:
mlflow.set_experiment("MlFlow-skinny test")

2026/03/18 08:36:16 INFO mlflow.tracking.fluent: Experiment with name 'MlFlow-skinny test' does not exist. Creating a new experiment.


<Experiment: artifact_location='', creation_time=1773819376146, experiment_id='45624b04-576f-4afc-916c-8788f2a617dd', last_update_time=None, lifecycle_stage='active', name='MlFlow-skinny test', tags={}>

## Train locally

Does MLFlow skinny connect with Azure MLFlow? Apparently it does much more. Autologging is still not that straightforward. 

In [ ]:
from sklearn.model_selection import train_test_split
import mltable
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

df = mltable.load("../../data/diabetes-data").to_pandas_dataframe()

X = df.drop(["PatientID","Diabetic"], axis=1).to_numpy()
y = np.where(df["Diabetic"].to_numpy(), 1, 0)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=7)

scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [29]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
from datetime import datetime as dt
from mlflow.models import infer_signature

timestamp_str = dt.now().strftime("%Y%m%d%H%M%S")

with mlflow.start_run(run_name=f"{timestamp_str}_local_run") as run:

    tree = RandomForestClassifier(n_estimators=80, criterion="gini")
    mlflow.log_params(tree.get_params())

    tree.fit(X_train_scaled, y_train)

    y_pred = tree.predict(X_test_scaled)
    accuracy = accuracy_score(y_test, y_pred)
    precision, recall, fscore, _ = precision_recall_fscore_support(y_test, y_pred)
    AUC = roc_auc_score(y_test, y_pred)

    mlflow.log_metrics({
         "accuracy" : accuracy
        ,"precision" : precision[1]
        ,"recall" : recall[1]
        ,"fscore" : fscore[1]
        ,"AUC" : AUC
    })

    signature = infer_signature(
         model_input    = X_test_scaled[:4,:]
        ,model_output   = y_pred[:4] 
        )
    
    mlflow.sklearn.log_model(tree, "model", signature=signature)


🏃 View run 20260318090707_local_run at: https://westeurope.api.azureml.ms/mlflow/v2.0/subscriptions/e46b15c2-60e4-419f-a352-ee019a3b0daf/resourceGroups/dg-associate-data-science/providers/Microsoft.MachineLearningServices/workspaces/aml-dg-associate-data-science/#/experiments/45624b04-576f-4afc-916c-8788f2a617dd/runs/4cb0a0b8-4eb7-4368-a362-11191f295eba
🧪 View experiment at: https://westeurope.api.azureml.ms/mlflow/v2.0/subscriptions/e46b15c2-60e4-419f-a352-ee019a3b0daf/resourceGroups/dg-associate-data-science/providers/Microsoft.MachineLearningServices/workspaces/aml-dg-associate-data-science/#/experiments/45624b04-576f-4afc-916c-8788f2a617dd


In [35]:
timestamp_str = dt.now().strftime("%Y%m%d%H%M%S")

with mlflow.start_run(run_name=f"{timestamp_str}_local_run_with_autolog") as run:

    mlflow.sklearn.autolog()

    tree = RandomForestClassifier(n_estimators=80, criterion="gini")
    tree.fit(X_train_scaled, y_train)

2026/03/18 09:22:01 WARNING mlflow.utils.autologging_utils: MLflow sklearn autologging is known to be compatible with 0.24.1 <= scikit-learn <= 1.6.1, but the installed version is 1.8.0. If you encounter errors during autologging, try upgrading / downgrading scikit-learn to a compatible version, or try upgrading MLflow.


🏃 View run 20260318092159_local_run_with_autolog at: https://westeurope.api.azureml.ms/mlflow/v2.0/subscriptions/e46b15c2-60e4-419f-a352-ee019a3b0daf/resourceGroups/dg-associate-data-science/providers/Microsoft.MachineLearningServices/workspaces/aml-dg-associate-data-science/#/experiments/45624b04-576f-4afc-916c-8788f2a617dd/runs/c55da3b0-0098-4a8e-84fe-184f3cda9469
🧪 View experiment at: https://westeurope.api.azureml.ms/mlflow/v2.0/subscriptions/e46b15c2-60e4-419f-a352-ee019a3b0daf/resourceGroups/dg-associate-data-science/providers/Microsoft.MachineLearningServices/workspaces/aml-dg-associate-data-science/#/experiments/45624b04-576f-4afc-916c-8788f2a617dd


## In Azure Machine Learning

### Create an MLFLow skinny environment

In [38]:
from azure.ai.ml.entities import Environment

environment_name = "diabetic_prediction_environment_mlflow_skinny"
environment_version = "0.0.2"

# Get the environment if available
try: 
    environment = ml_client.environments.get(name=environment_name, label="latest")
except: 
    environment = None

# Get the latest version of the environment
try:
    version = environment.version
except:
    version = None


if environment is None or version != environment_version:

    environment = Environment(
        image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu22.04"
        ,conda_file = "./src/environments/conda.yml"
        ,name = environment_name
        ,description = "Environment for diabetic classification - Associate Data Science Azure - MLFlow skinny"
        ,version = environment_version
    )

    print(f"Creating environment ... {environment_name}:{environment_version}")
    ml_client.create_or_update(environment)

else:
    environment = ml_client.environments.get(name=environment_name, label="latest")

Creating environment ... diabetic_prediction_environment_mlflow_skinny:0.0.2


### Run a job

In [48]:
from azure.ai.ml import command, Input
from azure.ai.ml.constants import AssetTypes
from datetime import datetime as dt

data_asset_name     = "diabetes-data-ml-table"
data_asset_version  = "0"

timestamp_str = dt.now().strftime("%Y%m%d%H%M%S")

command_job = command(
     description        = "Train a model and log the model afterwards as an mlflow model"
    ,display_name       = f"{timestamp_str}_aml_run"
    ,experiment_name    = "MlFlow-skinny test"
    ,environment        = "diabetic_prediction_environment_mlflow_skinny@latest"
    ,code               = "./src/code"
    ,command            = "python train.py --data-asset ${{inputs.data_asset}}"
    ,compute            = "diabetes-cluster"
    ,inputs             = {"data_asset" : Input(type=AssetTypes.MLTABLE, path=f"azureml:{data_asset_name}:{data_asset_version}")}
)

command_job_run = ml_client.create_or_update(command_job)

Uploading code (0.0 MBs): 100%|##########| 2255/2255 [00:00<00:00, 100078.89it/s]




In [58]:
from azure.ai.ml.entities import Model
from azure.ai.ml.constants import AssetTypes

command_job_run.name

run_model = Model(
     path           = f"azureml://jobs/{command_job_run.name}/outputs/artifacts/paths/model/"
    ,name           = "diabetes-model"
    ,description    = "Model created from run"
    ,type           = AssetTypes.MLFLOW_MODEL
    ,version        = "2"
)

ml_client.models.create_or_update(run_model)

Model({'job_name': 'ashy_snail_tlbjdpl7gl', 'intellectual_property': None, 'system_metadata': None, 'is_anonymous': False, 'auto_increment_version': False, 'auto_delete_setting': None, 'name': 'diabetes-model', 'description': 'Model created from run', 'tags': {}, 'properties': {}, 'print_as_yaml': False, 'id': '/subscriptions/e46b15c2-60e4-419f-a352-ee019a3b0daf/resourceGroups/dg-associate-data-science/providers/Microsoft.MachineLearningServices/workspaces/aml-dg-associate-data-science/models/diabetes-model/versions/2', 'Resource__source_path': '', 'base_path': 'c:\\Users\\dgouwy\\Documents\\devoProjects\\azure-machine-learning\\az-ml\\12_LoggingModelWithMLFlowSkinny', 'creation_context': <azure.ai.ml.entities._system_data.SystemData object at 0x0000013C9017BDD0>, 'serialize': <msrest.serialization.Serializer object at 0x0000013C8E246F90>, 'version': '2', 'latest_version': None, 'path': 'azureml://subscriptions/e46b15c2-60e4-419f-a352-ee019a3b0daf/resourceGroups/dg-associate-data-scien